In [ ]:
import requests
import sys

In [ ]:
sys.path.append('E:\wangzhilin\QuantumGalaxy')
from QGI.mysql import MYSQL
from datetime import date
import pandas as pd
import numpy as np
from QGI.feishu import *
from QGI.neoapi import Neo4j

In [ ]:
neo_uri = "neo4j+ssc://534ea9b7.databases.neo4j.io:7687"
neo_user = "neo4j"
neo_password = "QuantumGalaxy"
neo=Neo4j(neo_uri,neo_user,neo_password)

In [ ]:
neo_uri = "neo4j+ssc://08ef0a79.databases.neo4j.io"
neo_user = "QG_Editor"
neo_password = "editor"
neo=Neo4j(neo_uri,neo_user,neo_password)

In [ ]:
name='裂解反应'
node_cypher='match (n)-[*0..2]-(n1) where n.name="动力煤" return n1'

In [ ]:
def build_neo_data(name):
    '''从名字获取星图中两步以内相关的节点和边，返回组装的json'''
    node_cypher='match (n)-[*0..2]-(n1) where n.name="%s" return distinct n1'%name
    link_cypher='match (n)-[r]-(n1) where n.name="%s" return r \
        union \
        match (n)-[r1]-(n1)-[r]-(n2) where n.name="%s" return r'%(name,name)
    nodes=neo.simple_query(node_cypher)
    links=neo.simple_query(link_cypher)
    nodesdata=[]
    for record in nodes:
        node=record[0]
        nodesdata.append(node_process(node))
    linksdata=[]
    for record in links:
        link=record[0]
        linksdata.append(edge_process(link))
    return {'nodes':nodesdata,'links':linksdata}
    

In [9]:

def get_neo():
    
    neo_uri = "neo4j+ssc://534ea9b7.databases.neo4j.io:7687"
    neo_user = "neo4j"
    neo_password = "QuantumGalaxy"

    '''neo_uri = "neo4j+ssc://08ef0a79.databases.neo4j.io"
    neo_user = "QG_Editor"
    neo_password = "editor"'''


    neo=Neo4j(neo_uri,neo_user,neo_password)
    return neo
def write_node_type(nodetype):
    for k,v in {'Product':'产品','Technique':'工艺','Demand':'需求','Indicator':'指标','Company':'企业','Scenario':'情景'}.items():
        if k in nodetype:
            r=v
        
    try:       
        return r
    except:
        return 'unknown'
def write_edge_type(edgetype):
    for k,v in {'compose':'原材料','parent':'父节点','carry':'储运','measure':'反映','produce':'生产','peer':'同位','supply':'供应','imagine':'想象'}.items():
        if k in edgetype:
            r=v
        
    try:
        return r
    except:
        return 'unknown'
def node_process(node):
    dic={}
    dic['neoid']=node.id
    dic['category']=list(node.labels)
    dic['category']=write_node_type(dic['category'])
    prop=dict(node.items())
    dic['name']=prop.pop('name')
    #dic['others']=prop
    
    if prop.get('code',None):
        dic['code']=prop['code']
    if prop.get('value',None):
        dic['value']=prop['value']
    if prop.get('market_value',None):
        dic['market_value']=prop['market_value']
    
    if prop.get('chgrate1d',None):
        dic['chgrate1d']=prop['chgrate1d']
    if prop.get('stdchg1m',None):
        dic['stdchg1m']=prop['stdchg1m']
    if prop.get('stdchg3m',None):
        dic['stdchg3m']=prop['stdchg3m']

    ###dic中可能存在无法json化的数据 如Date类型###
    return dic

        
def edge_process(edge):
    dic={}
    dic['neoid']=edge.id
    dic['category']=edge.type
    dic['category']=write_edge_type(dic['category'])
    dic['source']=edge.start_node.id
    dic['target']=edge.end_node.id
    prop=dict(edge.items())
    
    #dic['others']=prop
    return dic
def build_neo_data(name):
    neo=get_neo()
    '''从名字获取星图中两步以内相关的节点和边，返回组装的json'''
    node_cypher='match (n)-[*0..2]-(n1) where n.name="%s" return distinct n1'%name
    link_cypher='match (n)-[r]-(n1) where n.name="%s" return r \
        union \
        match (n)-[r1]-(n1)-[r]-(n2) where n.name="%s" return r'%(name,name)
    nodes=neo.simple_query(node_cypher)
    links=neo.simple_query(link_cypher)
    nodesdata=[]
    for record in nodes:
        node=record[0]
        nodesdata.append(node_process(node))
    linksdata=[]
    for record in links:
        link=record[0]
        linksdata.append(edge_process(link))
    return {'nodes':nodesdata,'links':linksdata}
    

In [44]:
from werkzeug.datastructures import ImmutableMultiDict
d=ImmutableMultiDict([('exact_kw', ''), ('vague_kw', ''), ('code_kw', ''), ('expand_kws[]', '人工指标'), ('expand_kws[]', '基础货币')])

In [48]:
d.getlist('expand_kws[]')

['人工指标', '基础货币']

In [11]:
neo=get_neo()

In [13]:
r1=build_neo_data('裂解反应')

C:\Users\wangzhilin\AppData\Local\Temp\2\ipykernel_56128\184751864.py:34: DeprecationWarning: `id` is deprecated, use `element_id` instead
  dic['neoid']=node.id
C:\Users\wangzhilin\AppData\Local\Temp\2\ipykernel_56128\184751864.py:61: DeprecationWarning: `id` is deprecated, use `element_id` instead
  dic['neoid']=edge.id
C:\Users\wangzhilin\AppData\Local\Temp\2\ipykernel_56128\184751864.py:64: DeprecationWarning: `id` is deprecated, use `element_id` instead
  dic['source']=edge.start_node.id
C:\Users\wangzhilin\AppData\Local\Temp\2\ipykernel_56128\184751864.py:65: DeprecationWarning: `id` is deprecated, use `element_id` instead
  dic['target']=edge.end_node.id


In [14]:
r2=build_neo_data('基础货币')

C:\Users\wangzhilin\AppData\Local\Temp\2\ipykernel_56128\184751864.py:34: DeprecationWarning: `id` is deprecated, use `element_id` instead
  dic['neoid']=node.id
C:\Users\wangzhilin\AppData\Local\Temp\2\ipykernel_56128\184751864.py:61: DeprecationWarning: `id` is deprecated, use `element_id` instead
  dic['neoid']=edge.id
C:\Users\wangzhilin\AppData\Local\Temp\2\ipykernel_56128\184751864.py:64: DeprecationWarning: `id` is deprecated, use `element_id` instead
  dic['source']=edge.start_node.id
C:\Users\wangzhilin\AppData\Local\Temp\2\ipykernel_56128\184751864.py:65: DeprecationWarning: `id` is deprecated, use `element_id` instead
  dic['target']=edge.end_node.id


In [19]:
r1.update(r2)

In [35]:
name='基础货币'

In [43]:
s=[]
if s:
    print('what')

In [37]:
neo.simple_query('match (n)-[r]-(n1) where n.name="%s" return r \
        union \
        match (n)-[r1]-(n1)-[r]-(n2) where n.name="%s" return r'%(name,name))

[<Record r=<Relationship element_id='5:d2742333-9cc0-4a3e-8576-50684aa285cd:60' nodes=(<Node element_id='4:d2742333-9cc0-4a3e-8576-50684aa285cd:51' labels=frozenset() properties={}>, <Node element_id='4:d2742333-9cc0-4a3e-8576-50684aa285cd:23' labels=frozenset() properties={}>) type='measure' properties={}>>,
 <Record r=<Relationship element_id='5:d2742333-9cc0-4a3e-8576-50684aa285cd:25' nodes=(<Node element_id='4:d2742333-9cc0-4a3e-8576-50684aa285cd:23' labels=frozenset() properties={}>, <Node element_id='4:d2742333-9cc0-4a3e-8576-50684aa285cd:21' labels=frozenset() properties={}>) type='parent' properties={}>>,
 <Record r=<Relationship element_id='5:d2742333-9cc0-4a3e-8576-50684aa285cd:23' nodes=(<Node element_id='4:d2742333-9cc0-4a3e-8576-50684aa285cd:23' labels=frozenset() properties={}>, <Node element_id='4:d2742333-9cc0-4a3e-8576-50684aa285cd:28' labels=frozenset() properties={}>) type='parent' properties={}>>,
 <Record r=<Relationship element_id='5:d2742333-9cc0-4a3e-8576-50684a

In [24]:
neo.simple_query('match (n) where n.name contains"货币"  return n')

[<Record n=<Node element_id='4:d2742333-9cc0-4a3e-8576-50684aa285cd:12' labels=frozenset({'Product'}) properties={'name': '货币资金服务', 'backbone': '银行'}>>,
 <Record n=<Node element_id='4:d2742333-9cc0-4a3e-8576-50684aa285cd:22' labels=frozenset({'Product'}) properties={'name': '货币M2', 'backbone': '银行'}>>,
 <Record n=<Node element_id='4:d2742333-9cc0-4a3e-8576-50684aa285cd:23' labels=frozenset({'Product'}) properties={'name': '基础货币', 'backbone': '银行'}>>,
 <Record n=<Node element_id='4:d2742333-9cc0-4a3e-8576-50684aa285cd:24' labels=frozenset({'Product'}) properties={'code': '发行管理', 'name': '基础货币', 'backbone': '银行'}>>,
 <Record n=<Node element_id='4:d2742333-9cc0-4a3e-8576-50684aa285cd:32' labels=frozenset({'Product'}) properties={'name': '货币M1', 'backbone': '银行'}>>,
 <Record n=<Node element_id='4:d2742333-9cc0-4a3e-8576-50684aa285cd:51' labels=frozenset({'Indicator'}) properties={'code': '人工指标', 'name': '基础货币供应量', 'backbone': '银行'}>>]

In [34]:
neo.simple_query('WITH toInteger(rand() * 50) AS rand_node, 5 AS offset \
WITH range(rand_node - offset, rand_node + offset) AS rand_range \
MATCH (n)  \
WHERE id(n) IN rand_range \
RETURN n \
LIMIT 1')

[<Record n=<Node element_id='4:d2742333-9cc0-4a3e-8576-50684aa285cd:22' labels=frozenset({'Product'}) properties={'name': '货币M2', 'backbone': '银行'}>>]

In [21]:
r0={}
r0.update(r1)
r0.update(r2)
r0

{'nodes': [{'neoid': 23, 'category': '产品', 'name': '基础货币'},
  {'neoid': 51, 'category': '指标', 'name': '基础货币供应量', 'code': '人工指标'},
  {'neoid': 21, 'category': '产品', 'name': '存款准备金'},
  {'neoid': 28, 'category': '产品', 'name': '流通现金M0'},
  {'neoid': 20, 'category': '企业', 'name': '中国人民银行', 'code': '无'},
  {'neoid': 27, 'category': '产品', 'name': '银行间市场清算'},
  {'neoid': 24, 'category': '产品', 'name': '基础货币', 'code': '发行管理'},
  {'neoid': 38, 'category': '指标', 'name': '存款准备金率', 'code': '人工指标'},
  {'neoid': 26, 'category': '产品', 'name': '超额存款准备金'},
  {'neoid': 25, 'category': '产品', 'name': '法定存款准备金'},
  {'neoid': 43, 'category': '指标', 'name': 'M0供应量', 'code': '人工指标'},
  {'neoid': 32, 'category': '产品', 'name': '货币M1'},
  {'neoid': 30, 'category': '需求', 'name': '当前消费需求'},
  {'neoid': 22, 'category': '产品', 'name': '货币M2'}],
 'links': [{'neoid': 60, 'category': '反映', 'source': 51, 'target': 23},
  {'neoid': 25, 'category': '父节点', 'source': 23, 'target': 21},
  {'neoid': 23, 'category': '父节点', 'sourc

In [ ]:
name='裂解反应'
r=neo.simple_query('match (n)-[r]-(n1) where n.name="%s" return r \
        union \
        match (n)-[r1]-(n1)-[r]-(n2) where n.name="%s" return r'%(name,name))
r

In [ ]:
build_neo_data

In [ ]:
len(r)

In [ ]:
record=r[0]
record

In [ ]:
type(record[0])

In [ ]:
'compose' in record[0].type

In [ ]:
c=node_process(node)['catagory']

In [ ]:
'Product' in list(c)

In [ ]:
def write_node_type(edgetype):
    for k,v in {'Product':'产品','Indicator':'工艺','Demand':'需求','Indicator':'指标','Company':'企业'}.items():
        if k in edgetype:
            r=v
    return r

In [ ]:
link

In [ ]:
def edge_process(edge):
    dic={}
    dic['id']=edge.element_id
    dic['catagory']=edge.type
    dic['source']=edge.start_node.element_id
    dic['target']=edge.end_node.element_id
    prop=dict(edge.items())
    
    dic['others']=prop
    return dic
def node_process(node):
    dic={}
    dic['id']=node.element_id
    dic['catagory']=list(node.labels)
    prop=dict(node.items())
    dic['name']=prop.pop('name')
    dic['others']=prop
    return dic

In [ ]:
def work(name):
    

In [ ]:
nodes=[]
for record in r:
    node=record[0]
    nodes.append(node_process(node))

In [ ]:
nodes

In [ ]:
import json

In [ ]:
one=r[6]

In [ ]:
type(one[0])

In [ ]:
node=one[0]

In [ ]:
node.element_id

In [ ]:
node.labels

In [ ]:
dict(node.items())

In [ ]:
def get_neo():
    '''
    neo_uri = "neo4j+ssc://534ea9b7.databases.neo4j.io:7687"
    neo_user = "neo4j"
    neo_password = "QuantumGalaxy"'''
    neo_uri = "neo4j+ssc://08ef0a79.databases.neo4j.io"
    neo_user = "QG_Editor"
    neo_password = "editor"


    neo=Neo4j(neo_uri,neo_user,neo_password)
    return neo
def write_node_type(nodetype):
    for k,v in {'Product':'产品','Technique':'工艺','Demand':'需求','Indicator':'指标','Company':'企业'}.items():
        if k in nodetype:
            r=v
    try:    
        return r
    except:
        return 'unknown'
def write_edge_type(edgetype):
    for k,v in {'compose':'原材料','parent':'父节点','carry':'储运','measure':'反映','produce':'生产','peer':'同位'}.items():
        if k in edgetype:
            r=v
    try:
      
        return r
    except:
        return 'unknown'
def node_process(node):
    dic={}
    dic['neoid']=node.id
    dic['category']=list(node.labels)
    dic['category']=write_node_type(dic['category'])
    prop=dict(node.items())
    dic['name']=prop.pop('name')
    #dic['others']=prop

    if prop.get('code',None):
        dic['code']=prop['code']
    if prop.get('value',None):
        dic['value']=prop['value']
    if prop.get('chgrate1d',None):
        dic['chgrate1d']=prop['chgrate1d']
    if prop.get('stdchg1m',None):
        dic['stdchg1m']=prop['stdchg1m']
    if prop.get('stdchg3m',None):
        dic['stdchg3m']=prop['stdchg3m']
    ###dic中可能存在无法json化的数据 如Date类型###
    return dic

        
def edge_process(edge):
    dic={}
    dic['neoid']=edge.id
    dic['category']=edge.type
    dic['category']=write_edge_type(dic['category'])
    dic['source']=edge.start_node.id
    dic['target']=edge.end_node.id
    prop=dict(edge.items())
    
    #dic['others']=prop
    return dic
def build_neo_data(name):
    neo=get_neo()
    '''从名字获取星图中两步以内相关的节点和边，返回组装的json'''
    node_cypher='match (n)-[*0..2]-(n1) where n.name="%s" return distinct n1'%name
    link_cypher='match (n)-[r]-(n1) where n.name="%s" return r \
        union \
        match (n)-[r1]-(n1)-[r]-(n2) where n.name="%s" return r'%(name,name)
    nodes=neo.simple_query(node_cypher)
    links=neo.simple_query(link_cypher)
    nodesdata=[]
    for record in nodes:
        node=record[0]
        nodesdata.append(node_process(node))
    linksdata=[]
    for record in links:
        link=record[0]
        linksdata.append(edge_process(link))
    return {'nodes':nodesdata,'links':linksdata}
    

In [ ]:
build_neo_data('动力煤')

In [ ]:
name='动力煤'
node_cypher='match (n)-[*0..2]-(n1) where n.name="%s" return distinct n1'%name
link_cypher='match (n)-[r]-(n1) where n.name="%s" return r \
    union \
    match (n)-[r1]-(n1)-[r]-(n2) where n.name="%s" return r'%(name,name)
nodes=neo.simple_query(node_cypher)
links=neo.simple_query(link_cypher)

In [ ]:
d=nodes[1][0].items()
d=dict(d)
d

In [ ]:
d.get('code',None)

In [ ]:
'Product' in nodes[-1][0].labels

In [ ]:

try:
    sadf
except:
    print('hi')